In [ ]:
import boto3
import tarfile
import os

# Initialize S3 client
s3 = boto3.client('s3')

bucket_name = os.environ['READMISSION_S3_BUCKET']
model_key = os.environ['READMISSION_MODEL_S3_KEY']
local_tar_path = 'model.tar.gz'
extract_dir = './extracted_model'

# Download the archive from S3
s3.download_file(bucket_name, model_key, local_tar_path)

# Extract the archive to access raw weights
if tarfile.is_tarfile(local_tar_path):
    with tarfile.open(local_tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_dir)
    print(f"Model extracted to {extract_dir}")


In [ ]:
import tarfile
import joblib

# Use 'r:*' mode so Python auto-detects the compression format
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    tar.extractall()

# Now load your model (change file name to match yours)
model = joblib.load('model.pkl')


In [ ]:
model

In [ ]:
import pandas as pd
from botocore.config import Config

pd.set_option('display.max_columns', None)

# 1. Force a standard, isolated S3 client configuration
s3_client = boto3.client('s3', config=Config(signature_version='s3v4'))

# 2. Define your target bucket and object key path
bucket_name = os.environ['READMISSION_S3_BUCKET']
file_key = os.environ['READMISSION_FEATURES_S3_KEY']

# 3. Read the stream directly to bypass the JupyterLab automated discovery
response = s3_client.get_object(Bucket=bucket_name,
                                Key=file_key)
# 4. Load into your DataFrame
df = pd.read_csv(response['Body'])

df = df.drop(columns=[
    'stay_id'
],
    axis=1
)

age_bins = [0, 12, 19, 29, 39, 49, 59, 69, 120]
labels = ['0-12', '13-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70+']

df['age_group'] = pd.cut(
    df['age'],
    bins=age_bins,
    labels=labels,
    include_lowest=True
)
df = df.drop('age', axis=1)

df = pd.get_dummies(
    df,
    columns=['age_group'],
    dtype=int,
    drop_first=True
)

float_cols = df.select_dtypes(include=['float']).columns
df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast='float')

int_cols = df.select_dtypes(include=['integer']).columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')
df.shape

In [5]:
X = df.drop(columns=['readmit_30d', 'patient_id'])
y = df['readmit_30d']

In [6]:
from sklearn.metrics import classification_report, average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score

# Prevent the table from wrapping to a new line in the terminal
pd.set_option('display.width', None)

y_pred_proba = model.predict_proba(X)[:, 1]

print(y.info())

aucpr_score = average_precision_score(y, y_pred_proba)
brier_score = brier_score_loss(y, y_pred_proba)
roc_auc_score = roc_auc_score(y, y_pred_proba)
print(f"Test AUCPR Score: {aucpr_score:.4f}")
print(f"Test AUC-ROC Score: {roc_auc_score:.4f}")
print(f"Brier Score: {brier_score:.4f}\n")

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.5]
total_n = len(y)
results = []

for t in thresholds:
    y_pred_class = (y_pred_proba >= t).astype(int)
    report = classification_report(y, y_pred_class, output_dict=True)
    tn, fp, fn, tp = confusion_matrix(y, y_pred_class).ravel()

    results.append({
        "threshold": t,
        "precision": round(report['1']['precision'], 2),
        "recall": round(report['1']['recall'], 2),
        "f1": round(report['1']['f1-score'], 2),
        "support": int(report['1']['support']),
        "false pos (%)": f'{fp} ({(fp / total_n * 100):.1f}%)'
    })
    
threshold_summary = pd.DataFrame(results)
print(threshold_summary)

<class 'pandas.core.series.Series'>
RangeIndex: 11026 entries, 0 to 11025
Series name: readmit_30d
Non-Null Count  Dtype
--------------  -----
11026 non-null  int8 
dtypes: int8(1)
memory usage: 10.9 KB
None
Test AUCPR Score: 0.5387
Test AUC-ROC Score: 0.8676
Brier Score: 0.1552



   threshold  precision  recall    f1  support false pos (%)
0       0.10       0.27    0.99  0.43     1839  4818 (43.7%)
1       0.15       0.30    0.98  0.46     1839  4103 (37.2%)
2       0.20       0.33    0.96  0.49     1839  3559 (32.3%)
3       0.25       0.36    0.94  0.52     1839  3087 (28.0%)
4       0.30       0.38    0.93  0.54     1839  2827 (25.6%)
5       0.35       0.39    0.92  0.55     1839  2650 (24.0%)
6       0.40       0.40    0.91  0.55     1839  2537 (23.0%)
7       0.45       0.40    0.90  0.55     1839  2493 (22.6%)
8       0.50       0.40    0.90  0.56     1839  2445 (22.2%)


In [ ]:
"""
extract_model_metrics.py

Generates every metric table needed for the Tableau dashboards (technical +
business-translated) from a trained readmission model.

Assumes we already have, per model:
    - y_true       : array-like of actual 0/1 labels (readmitted_30d) on your test/holdout set
    - y_proba      : array-like of predicted probabilities from that model
    - encounter_id : array-like of encounter/patient IDs aligned with y_true/y_proba
    - X_test       : DataFrame of features used for prediction (needed for SHAP)
    - model        : the fitted model object (needed for SHAP)

Output: eight DataFrames, ready to write to CSV / S3 / RDS.
    1. summary_metrics_df   -> one row per model: AUC-PR, AUC-ROC, precision/recall @ threshold
    2. threshold_sweep_df   -> one row per (model, threshold): confusion matrix + precision/recall
    3. pr_curve_df          -> one row per (model, point) on the precision-recall curve
    4. shap_importance_df   -> one row per (model, feature): mean |SHAP| value, rank
    5. calibration_df       -> one row per (model, risk decile): predicted vs actual rate
    6. predictions_df       -> one row per (model, encounter_id): proba, actual, risk tier
    7. cost_df              -> one row per threshold: confusion matrix + $ cost/benefit
    8. monthly_df           -> one row per threshold: $ cost/benefit projected to real monthly volume
"""

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    brier_score_loss,
)

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False


# ---------------------------------------------------------------------------
# 1. Summary metrics
# ---------------------------------------------------------------------------
def compute_summary_metrics(y_true, y_proba, threshold=0.20):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    preds = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    return pd.DataFrame([{
        "auc_roc": roc_auc_score(y_true, y_proba),
        "auc_pr": average_precision_score(y_true, y_proba),
        "brier_score": brier_score_loss(y_true, y_proba),
        "threshold": threshold,
        "precision_at_threshold": precision,
        "recall_at_threshold": recall,
        "flagged_count": tp + fp,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
    }])


# ---------------------------------------------------------------------------
# 2. Threshold sweep
# ---------------------------------------------------------------------------
def threshold_sweep(y_true, y_proba, thresholds=np.arange(0.05, 0.95, 0.05)):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    total_n = len(y_true)
    rows = []
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else np.nan
        rows.append({
            "threshold": round(float(t), 2),
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "true_negative": tn,
            "precision": round(precision, 2) if not np.isnan(precision) else precision,
            "recall": round(recall, 2) if not np.isnan(recall) else recall,
            "f1": round(f1, 2) if not np.isnan(f1) else f1,
            "support": tp + fn,  # actual positives, matches classification_report's 'support'
            "flagged_count": tp + fp,
            "false_positive_pct": round(fp / total_n * 100, 1),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 3. Precision-recall curve points
# ---------------------------------------------------------------------------
def pr_curve_points(y_true, y_proba):
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_proba)
    return pd.DataFrame({
        "precision": precision_vals,
        "recall": recall_vals,
    })


# ---------------------------------------------------------------------------
# 4. SHAP feature importance
# ---------------------------------------------------------------------------
def shap_importance(model, X_test, top_n=15):
    if not SHAP_AVAILABLE:
        raise ImportError("pip install shap --break-system-packages")

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    # direction: does the feature push risk up or down on average when it's high?
    direction = np.sign(np.corrcoef(shap_values.T, X_test.values.T)[: X_test.shape[1], X_test.shape[1]:].diagonal())

    df = pd.DataFrame({
        "feature": X_test.columns,
        "mean_abs_shap": mean_abs_shap,
        "direction": direction,  # +1 = higher feature value raises risk, -1 = lowers it
    }).sort_values("mean_abs_shap", ascending=False).head(top_n)

    df["rank"] = range(1, len(df) + 1)
    return df


# ---------------------------------------------------------------------------
# 5. Calibration table
# ---------------------------------------------------------------------------
def calibration_table(y_true, y_proba, n_bins=10):
    df = pd.DataFrame({"y_true": np.asarray(y_true), "y_proba": np.asarray(y_proba)})
    df["decile"] = pd.qcut(df["y_proba"], n_bins, labels=False, duplicates="drop")

    cal = (
        df.groupby("decile")
        .agg(
            predicted_mean=("y_proba", "mean"),
            actual_rate=("y_true", "mean"),
            count=("y_true", "size"),
        )
        .reset_index()
    )

    return cal


# ---------------------------------------------------------------------------
# 6. Row-level predictions with risk tiers
# ---------------------------------------------------------------------------
def assign_risk_tier(p, low_cutoff=0.10, high_cutoff=0.20):
    """Cutoffs default to bracket your 0.20 decision threshold -- adjust to taste."""
    if p < low_cutoff:
        return "Low"
    elif p < high_cutoff:
        return "Medium"
    return "High"


def predictions_table(encounter_id, y_true, y_proba, threshold=0.20, dates=None):
    df = pd.DataFrame({
        "encounter_id": encounter_id,
        "actual_label": np.asarray(y_true),
        "predicted_proba": np.asarray(y_proba),
    })
    df["predicted_label"] = (df["predicted_proba"] >= threshold).astype(int)
    df["risk_tier"] = df["predicted_proba"].apply(assign_risk_tier)
    if dates is not None:
        df["year"] = pd.to_datetime(pd.Series(dates).reset_index(drop=True)).dt.year
    return df


# ---------------------------------------------------------------------------
# 7. Cost-based threshold optimization
#
#    Net savings at a given threshold =
#        (expected value of readmissions prevented) - (cost of intervening
#        on everyone flagged)
#
#    Net Savings = intervention_efficacy * avg_readmission_cost
#                  * sum(predicted_proba for all flagged patients)
#                  - flagged_count * cost_per_intervention
# ---------------------------------------------------------------------------
def cost_based_threshold_analysis(
    preds_df,
    cost_per_intervention,
    avg_readmission_cost,
    intervention_efficacy,
    thresholds=np.arange(0.05, 0.55, 0.05),
    avg_monthly_admissions=None,
):
    y_true = preds_df["actual_label"].to_numpy()
    proba = preds_df["predicted_proba"].to_numpy()
    n_holdout = len(proba)

    rows = []

    for t in thresholds:
        preds = (proba >= t).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

        flagged = preds == 1
        flagged_count = int(flagged.sum())

        expected_prevented_value = (
            intervention_efficacy
            * avg_readmission_cost
            * proba[flagged].sum()
        )

        intervention_cost = flagged_count * cost_per_intervention
        net_savings = expected_prevented_value - intervention_cost

        # How many admissions this threshold is expected to actually prevent.
        # Compare against avg_readmission_cost for the "$900 to
        # avoid a $16,300 readmission" framing.
        expected_admissions_prevented = (
            expected_prevented_value / avg_readmission_cost if avg_readmission_cost > 0 else np.nan
        )
        cost_per_saved_admission = (
            intervention_cost / expected_admissions_prevented
            if expected_admissions_prevented and expected_admissions_prevented > 0
            else np.nan
        )

        row = {
            "threshold": round(float(t), 2),

            # Confusion matrix
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "true_negative": tn,

            # Counts
            "flagged_count": flagged_count,

            # Cost metrics
            "expected_prevented_value": expected_prevented_value,
            "intervention_cost": intervention_cost,
            "net_savings": net_savings,
            "expected_admissions_prevented": (
                round(expected_admissions_prevented, 1)
                if expected_admissions_prevented == expected_admissions_prevented
                else np.nan
            ),
            "cost_per_saved_admission": cost_per_saved_admission,
        }

        if avg_monthly_admissions is not None:
            expected_value_rate = proba[flagged].sum() / n_holdout
            row["monthly_expected_admissions_prevented"] = round(
                intervention_efficacy * expected_value_rate * avg_monthly_admissions, 1
            )

        rows.append(row)

    return pd.DataFrame(rows)


def best_threshold_by_savings(cost_df):
    """Returns the single row (threshold + all its cost columns) with the highest net savings."""
    return cost_df.loc[cost_df["net_savings"].idxmax()]


# ---------------------------------------------------------------------------
# 8. Monthly savings projection
# ---------------------------------------------------------------------------
def estimate_avg_monthly_admissions(admission_dates):
    """
    admission_dates: array-like of admission/encounter dates across your
    FULL dataset (not just the holdout set) -- e.g. df['admission_date'].
    Returns the average number of admissions per calendar month.
    """
    dates = pd.to_datetime(pd.Series(admission_dates))
    return dates.dt.to_period("M").value_counts().mean()


def estimate_avg_monthly_admissions_by_years(stay_end_dates, start_year=2021, end_year=2025):
    """
    stay_end_dates: array-like of stay-end dates across your FULL dataset
    (e.g. df['stay_end']). Filters to YEAR(stay_end) in [start_year, end_year]
    inclusive, then returns total admissions in that window / (num years * 12)
    -- a fixed multi-year monthly average based on discharge year, rather
    than averaging individual calendar-month counts the way
    estimate_avg_monthly_admissions() does.
    """
    dates = pd.to_datetime(pd.Series(stay_end_dates))
    mask = dates.dt.year.between(start_year, end_year)
    total_admissions = int(mask.sum())
    n_months = (end_year - start_year + 1) * 12
    return total_admissions / n_months


def project_monthly_savings(
    preds_df,
    cost_per_intervention,
    avg_readmission_cost,
    intervention_efficacy,
    avg_monthly_admissions,
    thresholds=np.arange(0.05, 0.55, 0.05),
):
    proba = preds_df["predicted_proba"].to_numpy()
    n_holdout = len(proba)

    rows = []
    for t in thresholds:
        flagged = proba >= t

        flagged_rate = flagged.sum() / n_holdout
        expected_value_rate = proba[flagged].sum() / n_holdout

        monthly_flagged_count = flagged_rate * avg_monthly_admissions
        monthly_prevented_value = intervention_efficacy * avg_readmission_cost * expected_value_rate * avg_monthly_admissions
        monthly_intervention_cost = monthly_flagged_count * cost_per_intervention
        monthly_net_savings = monthly_prevented_value - monthly_intervention_cost

        rows.append({
            "threshold": round(float(t), 2),
            "monthly_flagged_count": round(monthly_flagged_count, 1),
            "monthly_prevented_value": monthly_prevented_value,
            "monthly_intervention_cost": monthly_intervention_cost,
            "monthly_net_savings": monthly_net_savings,
            "annualized_net_savings": monthly_net_savings * 12,
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 9. Aggregated cost breakdown for a specific year range.
#
#    cost_per_saved_admission = intervention_cost / expected_admissions_prevented
#    where expected_admissions_prevented = expected_prevented_value / avg_readmission_cost
# ---------------------------------------------------------------------------
def cost_analysis_for_year_range(
    preds_df,
    threshold,
    cost_per_intervention,
    avg_readmission_cost,
    intervention_efficacy,
    start_year=2021,
    end_year=2025,
):
    if "year" not in preds_df.columns:
        raise ValueError(
            "preds_df has no 'year' column -- pass dates= into predictions_table() "
            "when you build preds_df so each row is tagged with an admission year."
        )

    range_df = preds_df[(preds_df["year"] >= start_year) & (preds_df["year"] <= end_year)]

    proba = range_df["predicted_proba"].to_numpy()
    flagged = proba >= threshold
    flagged_count = int(flagged.sum())

    expected_prevented_value = intervention_efficacy * avg_readmission_cost * proba[flagged].sum()
    intervention_cost = flagged_count * cost_per_intervention

    expected_admissions_prevented = (
        expected_prevented_value / avg_readmission_cost if avg_readmission_cost > 0 else np.nan
    )
    cost_per_saved_admission = (
        intervention_cost / expected_admissions_prevented
        if expected_admissions_prevented and expected_admissions_prevented > 0
        else np.nan
    )

    return pd.DataFrame([{
        "year_range": f"{start_year}-{end_year}",
        "threshold": threshold,
        "patient_count": len(range_df),
        "flagged_count": flagged_count,
        "intervention_cost": intervention_cost,
        "expected_prevented_value": expected_prevented_value,
        "expected_admissions_prevented": round(expected_admissions_prevented, 1),
        "cost_per_saved_admission": cost_per_saved_admission,
    }])


# Prevent the table from wrapping to a new line in the terminal
pd.set_option("display.width", None)

y_pred_proba = model.predict_proba(X)[:, 1]

aucpr_score = average_precision_score(y, y_pred_proba)
brier_score = brier_score_loss(y, y_pred_proba)
auc_roc = roc_auc_score(y, y_pred_proba)
print(f"Test AUCPR Score: {aucpr_score:.4f}")
print(f"Test AUC-ROC Score: {auc_roc:.4f}")
print(f"Brier Score: {brier_score:.4f}\n")

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.5]

summary_df = compute_summary_metrics(y, y_pred_proba, threshold=0.10)
sweep_df = threshold_sweep(y, y_pred_proba, thresholds=thresholds)
pr_df = pr_curve_points(y, y_pred_proba)
cal_df = calibration_table(y, y_pred_proba)
shap_df = shap_importance(model, X)

ADMISSION_DATES = None  # e.g. df.loc[X.index, "admission_date"]

preds_df = predictions_table(X.index, y, y_pred_proba, threshold=0.10, dates=ADMISSION_DATES)

print(sweep_df[["threshold", "precision", "recall", "f1", "support", "false_positive_pct"]])

# --- cost-based threshold analysis ---
COST_PER_INTERVENTION = 300         
AVG_READMISSION_COST = 16300        
INTERVENTION_EFFICACY = 0.2         

STAY_END_DATES = None  # e.g. df["stay_end"]

if STAY_END_DATES is not None:
    AVG_MONTHLY_ADMISSIONS_2021_2025 = estimate_avg_monthly_admissions_by_years(
        STAY_END_DATES, start_year=2021, end_year=2025
    )
else:
    AVG_MONTHLY_ADMISSIONS_2021_2025 = 450 

cost_df = cost_based_threshold_analysis(
    preds_df,
    cost_per_intervention=COST_PER_INTERVENTION,
    avg_readmission_cost=AVG_READMISSION_COST,
    intervention_efficacy=INTERVENTION_EFFICACY,
    avg_monthly_admissions=AVG_MONTHLY_ADMISSIONS_2021_2025,
)
best_row = best_threshold_by_savings(cost_df)
print(f"\nBest threshold by net savings: {best_row['threshold']} "
      f"(net savings: ${best_row['net_savings']:,.0f})")
print(cost_df[[
    "threshold",
    "true_positive",
    "false_positive",
    "net_savings",
    "cost_per_saved_admission",
    "monthly_expected_admissions_prevented",
]])

# --- monthly-volume projection ---
AVG_MONTHLY_ADMISSIONS = 500  # replace with your actual computed value

monthly_df = project_monthly_savings(
    preds_df,
    cost_per_intervention=COST_PER_INTERVENTION,
    avg_readmission_cost=AVG_READMISSION_COST,
    intervention_efficacy=INTERVENTION_EFFICACY,
    avg_monthly_admissions=AVG_MONTHLY_ADMISSIONS,
)
best_monthly_row = best_threshold_by_savings(
    monthly_df.rename(columns={"monthly_net_savings": "net_savings"})
)
print(f"\nBest threshold by projected monthly savings: {best_monthly_row['threshold']} "
      f"(est. monthly savings: ${best_monthly_row['net_savings']:,.0f}, "
      f"est. annual savings: ${best_monthly_row['net_savings'] * 12:,.0f})")
print(monthly_df[["threshold", "monthly_flagged_count", "monthly_net_savings", "annualized_net_savings"]])

if "year" in preds_df.columns:
    year_range_df = cost_analysis_for_year_range(
        preds_df,
        threshold=best_row["threshold"],
        cost_per_intervention=COST_PER_INTERVENTION,
        avg_readmission_cost=AVG_READMISSION_COST,
        intervention_efficacy=INTERVENTION_EFFICACY,
        start_year=2021,
        end_year=2025,
    )
    print("\nAggregated cost breakdown (2021-2025):")
    print(year_range_df[["year_range", "patient_count", "intervention_cost", "cost_per_saved_admission"]])
    year_range_df.to_csv("cost_analysis_2021_2025.csv", index=False)
else:
    print("\nSkipping 2021-2025 cost breakdown -- set ADMISSION_DATES above to enable it.")

summary_df.to_csv("summary_metrics.csv", index=False)
sweep_df.to_csv("threshold_sweep.csv", index=False)
cost_df.to_csv("threshold_cost_analysis.csv", index=False)
monthly_df.to_csv("threshold_monthly_projection.csv", index=False)
pr_df.to_csv("pr_curve.csv", index=False)
cal_df.to_csv("calibration.csv", index=False)
shap_df.to_csv("shap_importance.csv", index=False)
preds_df.to_csv("predictions.csv", index=False)